# TTS(Text To Speech)
- 글자를 소리(음성)로 바꾸는 기술
- 예시
    - 네비게이션: "500m 앞 우회전입니다~" 라고 말해줌
    - 스마트 스피커: "오늘 날씨는 맑고 기온은 25도입니다."
    - 전자책 앱: 책 내용을 음성으로 읽어줌

- 강의 자료 링크 : https://www.notion.so/01-15e117802625814ba504e4a45f41d9a4?source=copy_link

## 1. gTTS

- Google Text-to-Speech를 간단히 호출해 텍스트를 MP3 음성으로 변환하는 파이썬 라이브러리
- 설치와 사용법이 쉬워 실습용으로 적합하지만, 음성 톤·스타일 세부 제어와 커스텀 보이스 기능은 제한적

### 1) gTTS 패키지 설치
- Google Text-to-Speech API를 사용하여 텍스트를 자연스러운 음성으로 변환하는 파이썬 라이브러리
- 구글 번역기에서 체험 가능 : https://translate.google.co.kr/

In [1]:
%pip install gTTS

Note: you may need to restart the kernel to use updated packages.


### 2) 영어 문장 사운드 파일 생성 및 저장
- gTTS 클래스는 텍스트를 음성으로 변환하는 주요 클래스이다.

In [2]:
from gtts import gTTS

In [3]:
# 영어 문장
text = 'Hello, nice to meet you. Today is a great day!'
file_name = 'sample.mp3'
tts_en = gTTS(text=text, lang='en')
tts_en.save(file_name)

### 3) .mp3 파일 재생

In [4]:
from IPython.display import Audio, display

sound = display(Audio(file_name, autoplay=True))

### 4) 한글 문장 TTS


- gTTS 주요 옵션

| 속성         | 설명                                     | 타입  | 기본값    |
|--------------|------------------------------------------|--------|------------|
| `text`       | 음성으로 변환할 텍스트                  | `str` | **필수**   |
| `lang`       | 언어 코드 (예: `"ko"`, `"en"`, `"ja"`, `"fr"`) | `str` | `"en"`     |
| `slow`       | 음성 속도 (느리게 출력 여부)             | `bool` | `False`    |
| `lang_check` | 언어 코드 유효성 확인 여부               | `bool` | `True`     |
| `tld`        | Google TTS 서버의 Top-Level Domain<br>(글로벌: `com`, 한국: `co.kr`) | `str` | `"com"`     |


In [5]:
# 한글 문장
text = '안녕하세요. 반가워요. 오늘도 화이팅입니다.'
file_name = 'sample2.mp3'
tts_ko = gTTS(text=text, lang='ko')
tts_ko.save(file_name)

sound = display(Audio(file_name, autoplay=True))

### 5) 텍스트 파일로부터 TTS

In [6]:
# 텍스트 파일로부터 읽기
sample_text = """
간장공장 공장장은 강 공장장이고
된장공장 공장장은 장 공장장이다.
내가 그린 기린 그림은 니가 그린 기림 그림이고
니가 그린 기린 그림은 내가 그린 기린 그림이다.
"""


# sample3.txt라는 이름으로 저장
with open('sample3.txt', 'w', encoding='utf-8') as f:
    f.write(sample_text)


In [7]:
with open('sample3.txt', 'r', encoding='utf-8') as f:
    file_text = f.read()

tts = gTTS(text=file_text, lang='ko')
tts.save('sample3.mp3')

display(Audio('sample.mp3', autoplay=True))

## 2. Openai API

- **gpt-4o-mini-tts**는 OpenAI의 TTS 모델로, 텍스트를 자연스러운 음성으로 생성하고 `voice`, `speed`, `instructions`로 톤을 제어할 수 있습니다.
- 짧은 안내 음성부터 감정이 들어간 대화형 음성까지 API로 빠르게 만들 수 있습니다.

In [8]:
# https://www.openai.fm/ 목소리 탐험

In [9]:
from dotenv import load_dotenv 
load_dotenv()

True

In [10]:
from openai import OpenAI
client = OpenAI()

In [11]:
from pathlib import Path

output_path = Path('./audio/speech.mp3')
output_path.parent.mkdir(parents=True, exist_ok=True)

with client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='coral',
    input='안녕하세요. 오늘 비가 많이 옵니다. 날씨가 꿀꿀해요.',
    speed=0.3, # 0.25 ~ 4.0
    instructions='슬픈 목소리로 말해줘.'
) as response:
    response.stream_to_file(output_path)


In [12]:
# uv add pydud pyaudioop pyaudio
# 오디오 백그라운드 실행

display(Audio("./audio/speech.mp3", autoplay=True))


## 3. ElevenLabs API

- **eleven_multilingual_v2**는 다국어 음성 합성과 보이스 클로닝에 강점이 있는 ElevenLabs TTS 모델
- `stability`, `similarity_boost` 같은 설정으로 감정 변화와 원본 화자 유사도를 조절할 수 있음

In [13]:
# 로그인 https://elevenlabs.io/ 
# 라이브러리 설치 uv add elevenlabs

In [14]:
from dotenv import load_dotenv 
load_dotenv()

True

In [15]:
%pip install elevenlabs

import os
from pathlib import Path
from dotenv import load_dotenv
from elevenlabs.client import ElevenLabs

env_path = Path.cwd() / '.env'
load_dotenv(dotenv_path=env_path, override=False)

api_key = os.getenv('ELEVENLABS_API_KEY')
if not api_key:
    raise ValueError('ELEVENLABS_API_KEY가 .env에서 찾을 수 없습니다.')

client = ElevenLabs(api_key=api_key)


Note: you may need to restart the kernel to use updated packages.


In [16]:
response = client.voices.search()

for voice in response.voices:
    print(voice)


voice_id='ZqvIIuD5aI9JFejebHiH' name='Mira - Meditation, Calming Down, Relaxing' samples=None category='professional' fine_tuning=FineTuningResponse(is_allowed_to_fine_tune=True, state={'eleven_v2_5_flash': 'fine_tuned', 'eleven_flash_v2': 'fine_tuned', 'eleven_turbo_v2_5': 'fine_tuned', 'eleven_v2_flash': 'fine_tuned', 'eleven_multilingual_sts_v2': 'fine_tuned', 'eleven_flash_v2_5': 'fine_tuned', 'eleven_multilingual_v2': 'fine_tuned', 'eleven_turbo_v2': 'fine_tuned'}, verification_failures=[], verification_attempts_count=0, manual_verification_requested=False, language='en', progress={}, message={'eleven_v2_5_flash': '', 'eleven_flash_v2': '', 'eleven_turbo_v2_5': '', 'eleven_v2_flash': '', 'eleven_multilingual_sts_v2': '', 'eleven_flash_v2_5': '', 'eleven_multilingual_v2': '', 'eleven_turbo_v2': ''}, dataset_duration_seconds=None, verification_attempts=None, slice_ids=None, manual_verification=None, max_verification_attempts=0, next_max_verification_attempts_reset_unix_ms=0, finetun

In [17]:
# https://elevenlabs.io/app/voice-library

In [18]:
from elevenlabs import VoiceSettings
# 음성 변환
audio = client.text_to_speech.convert(
    text="할 수 있습니다. 우리는 최고입니다. 에스케이 네트웍스 30기 화이팅",
    voice_id="hpp4J3VqNfWAUOO0d1Us",
    model_id="eleven_multilingual_v2",
    output_format="mp3_44100_128", 
    voice_settings=VoiceSettings(
        stability=0.3,              # 감정의 변화 정도(낮을수록 감정 풍부)
        similarity_boost=0.8,       # 원 화자와의 유사도(높을수록 비슷한 목소리)
        use_speaker_boost=True      # 화자 억양 향상
    )
)

file_path = "./audio/voice_clone_Bella.mp3"

audio_bytes = b"".join(audio)

with open(file_path, "wb") as f:
    f.write(audio_bytes)

In [ ]:
# 오디오 백그라운드 실행
display(Audio("./audio/voice_clone_Bella.mp3", autoplay=True))


---

## [실습]
1. 같은 문장, 다른 언어로 합성하기
2. 3문장 이상 들어 있는 `.txt` 파일을 만들고, 내일 내용을 읽어 음성으로 변환하기
3. 음성 속도 비교하기. OpenAI TTS의 `speed` 값을 느림, 보통, 빠름으로 바꿔 같은 문장을 3개 파일로 저장하고 들어보기
4. 감정이 다른 음성 만들기. OpenAI TTS의 `instructions`를 활용.
5. 미니 오디오북 제작하기. 5문장 이상의 짧은 이야기나 설명문을 작성하고, 문장별로 자연스럽게 끊어 읽히도록 음성 파일을 제작.
6. 주제를 하나 정해 30초 내외의 음성 콘텐츠를 제작. 예: 뉴스 브리핑, 상품 소개, 게임 캐릭터 대사, 날씨 안내. 사용한 모델과 설정값을 함께 정리.

In [20]:
from pathlib import Path
from openai import OpenAI

openai_client = OpenAI()

text = '제발 나를 건드리지 말고 가만히 내버려두세요. 당신은 당신 할 것을 하세요!'
speeds = [0.25, 0.5, 1.0, 2.0, 4.0]
output_dir = Path('./audio')
output_dir.mkdir(parents=True, exist_ok=True)

for speed in speeds:
    output_path = output_dir / f'speech_{speed}.mp3'

    with openai_client.audio.speech.with_streaming_response.create(
        model='gpt-4o-mini-tts',
        voice='coral',
        input=text,
        speed=speed,
        instructions='명랑한 목소리로 말해줘.'
    ) as response:
        response.stream_to_file(output_path)

In [23]:
from pathlib import Path
from openai import OpenAI

openai_client = OpenAI()

text = '''서초구 AI 부트캠프의 강의실은 심야의 냉기와 열기로 가득 차 있었다.
화면을 채운 복잡한 데이터 스트림 속에서, 백만장자라는 목표는 감정이 아닌 정교한 수식으로 치환된다.
알고리즘의 오차를 줄여가는 매 순간은 자본의 논리를 학습하는 과정이었다.
새벽 3시의 적막을 깨고 실행된 코드는 부의 흐름을 재편할 준비를 마친다.
모니터 너머로 비치는 그의 눈빛에는 막연한 희망 대신 확신에 찬 계산만이 남아 있었다.'''

output_dir = Path('./audio')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'speech_1.5.mp3'

with openai_client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='echo',
    input=text,
    speed=1.5,
    instructions='록커가 샤우팅 하듯이 말해줘. 그리고 문장별로 끊어서 말해주길 바랄게. '
) as response:
    response.stream_to_file(output_path)

In [24]:
display(Audio("./audio/speech_1.5.mp3", autoplay=True))